# Shingling: Finding Reuse Without Reading

**Computational Text Analysis Pack · Module 18 — Text Reuse and Intertextuality**

*Technique:* Character and word shingles; Jaccard similarity; the quadratic wall

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/08-text-reuse-and-networks/18a_shingling_and_jaccard.ipynb)

Data: `masakhanews.csv`, `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Before the wire services, a story spread by being **reprinted** — copied,
abridged, and retitled as it moved from paper to paper. Finding those chains is
one of the most productive things distant reading does with a newspaper archive,
and it is the method behind the Viral Texts project and its African and Finnish
counterparts.

The problem is that two copies of a story are almost never identical: OCR noise,
editorial cuts, a changed headline. So the method cannot be string matching. It
is **shingling** — cut each document into overlapping fragments, and measure how
many fragments two documents share.

## The tooling

**Passim** (Smith, Cordell et al.) is the standard at archive scale — Spark-based
and deliberately tolerant of OCR noise. **TextPAIR** (ARTFL/Chicago) combines
n-gram matching with sequence alignment; **TRACER** (Büchler) is highly
configurable and strong on ancient languages; R's **textreuse** (Mullen)
implements the shingling/MinHash pipeline we are about to build. Some projects
reach for **BLAST**, the bioinformatics aligner, which works because DNA
alignment and text reuse are the same problem.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
def shingles(text, k=5, unit="word"):
    """Overlapping k-grams. Word shingles for prose, character for noisy OCR."""
    if unit == "word":
        toks = tokenise(text)
        return {" ".join(toks[i:i + k]) for i in range(len(toks) - k + 1)}
    s = " ".join(tokenise(text))
    return {s[i:i + k] for i in range(len(s) - k + 1)}

def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

a = "the governor arrived at the port on tuesday morning with his retinue"
b = "The Governor arrived at the port on Tuesday morning, with his retinue."
c = "the harvest this season has been poor across the northern province"
print(f"near-identical  : {jaccard(shingles(a, 3), shingles(b, 3)):.3f}")
print(f"unrelated       : {jaccard(shingles(a, 3), shingles(c, 3)):.3f}")
print()
print("Case and punctuation vanish in tokenising, so the reprint scores high")
print("without any of the strings being equal.")

## Why shingle size is the whole method

Too small and every document shares fragments with every other. Too large and a
single OCR error breaks the match. The choice is a research decision, so it
should be made by looking at what it does rather than by copying a default.

In [ ]:
# Build a controlled test: real documents, plus deliberately damaged copies.
import random
rng = random.Random(5)
originals = df["text"].head(60).tolist()

def damage(text, rate):
    chars = list(text)
    for i in range(len(chars)):
        if rng.random() < rate:
            chars[i] = rng.choice("aeionrst ")
    return "".join(chars)

print(f"{'k':>3}{'exact copy':>13}{'2% noise':>11}{'8% noise':>11}{'unrelated':>12}")
print("-" * 50)
for k in [2, 3, 5, 8, 12]:
    same, noisy2, noisy8, diff = [], [], [], []
    for i, doc in enumerate(originals[:25]):
        s = shingles(doc, k)
        same.append(jaccard(s, shingles(doc, k)))
        noisy2.append(jaccard(s, shingles(damage(doc, 0.02), k)))
        noisy8.append(jaccard(s, shingles(damage(doc, 0.08), k)))
        diff.append(jaccard(s, shingles(originals[(i + 7) % 25], k)))
    print(f"{k:>3}{np.mean(same):>13.3f}{np.mean(noisy2):>11.3f}"
          f"{np.mean(noisy8):>11.3f}{np.mean(diff):>12.3f}")
print()
print("The number that matters is the GAP between the noisy-copy column and the")
print("unrelated column. That gap is your detection margin, and it closes as k")
print("grows. For noisy OCR, small word shingles or character shingles win.")

In [ ]:
plt.figure(figsize=(7, 4))
ks = [2, 3, 5, 8, 12]
for label, rate, style in [("2% noise", 0.02, "-o"), ("8% noise", 0.08, "-s"),
                           ("unrelated", None, "--^")]:
    ys = []
    for k in ks:
        vals = []
        for i, doc in enumerate(originals[:25]):
            s = shingles(doc, k)
            other = (shingles(damage(doc, rate), k) if rate
                     else shingles(originals[(i + 7) % 25], k))
            vals.append(jaccard(s, other))
        ys.append(np.mean(vals))
    plt.plot(ks, ys, style, label=label)
plt.xlabel("shingle size k (words)")
plt.ylabel("mean Jaccard similarity")
plt.title("The detection margin narrows as k grows")
plt.legend()
plt.grid(alpha=.3)
plt.tight_layout()

## Finding real reuse

Now on actual documents, with no synthetic damage — pairs the corpus contains
because news is syndicated and stories get rewritten.

In [ ]:
docs = df.sample(n=350, random_state=8).reset_index(drop=True)
sig = [shingles(t, 4) for t in docs["text"]]

pairs = []
for i in range(len(sig)):
    for j in range(i + 1, len(sig)):
        s = jaccard(sig[i], sig[j])
        if s > 0.10:
            pairs.append((s, i, j))
pairs.sort(reverse=True)
print(f"{len(pairs)} pairs above 0.10 similarity, out of "
      f"{len(sig) * (len(sig) - 1) // 2:,} comparisons\n")
for s, i, j in pairs[:5]:
    print(f"similarity {s:.3f}   [{docs.loc[i, 'language']}] / [{docs.loc[j, 'language']}]")
    print(f"   A: {docs.loc[i, 'headline'][:78]}")
    print(f"   B: {docs.loc[j, 'headline'][:78]}")
    print()

## The wall

Notice the comparison count above. Similarity is pairwise, so the work grows as
n². Three hundred documents is sixty thousand comparisons and takes a moment;
a hundred thousand documents is five billion and does not finish.

Every tool named at the top of this notebook exists to get around that number.
The next notebook builds the standard escape.

In [ ]:
print(f"{'documents':>12}{'comparisons':>18}{'time at 50k/sec':>20}")
print("-" * 50)
for n in [300, 3_000, 30_000, 300_000, 3_000_000]:
    comps = n * (n - 1) // 2
    secs = comps / 50_000
    unit = (f"{secs:.0f} s" if secs < 90 else
            f"{secs/60:.0f} min" if secs < 5400 else
            f"{secs/3600:.0f} hours" if secs < 172800 else
            f"{secs/86400:,.0f} days")
    print(f"{n:>12,}{comps:>18,}{unit:>20}")

### 🔧 Your turn 1

Switch `shingles` to character shingles with k=9 and re-run the noise table. Are they more robust than word shingles?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Character shingles degrade more gracefully under noise, because a corrupted
character destroys only the handful of shingles overlapping it, whereas a
corrupted character destroys every word shingle containing that word. That is
why Passim and the Finnish newspaper work lean on character n-grams. The cost is
volume: character shingles are far more numerous, so both memory and comparison
time rise, which makes the next notebook's technique more necessary rather than
less.

</details>